# Verificación de la capa Silver — Iceberg

Revisamos la tabla `nessie.silver.yellow_taxi` tras la limpieza:
referencias Nessie, metadatos Iceberg, snapshots, particionado y calidad de datos.
También comparamos contra bronze para validar las reglas de limpieza.

In [1]:
from common.spark_factory import get_spark
from common import nessie_manager as nessie

spark = get_spark("verify-silver-iceberg")
spark.version

26/04/19 17:49:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/19 17:49:52 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/04/19 17:49:54 WARN S3ABlockOutputStream: Application invoked the Syncable API against stream writing to events/app-20260419174951-0012.inprogress. This is unsupported


'3.5.8'

## 1. Catálogo Nessie — Referencias y namespaces

In [2]:
print("Nessie references:\n")
nessie.print_references(spark)

refs_df = spark.sql("LIST REFERENCES IN nessie")
type_col = "refType" if "refType" in refs_df.columns else "type"
branches = refs_df.filter(refs_df[type_col].ilike("branch")).count()
tags     = refs_df.filter(refs_df[type_col].ilike("tag")).count()
print(f"\n  Branches: {branches}  |  Tags: {tags}")

Nessie references:



26/04/19 17:50:02 INFO CatalogUtil: Loading custom FileIO implementation: org.apache.iceberg.hadoop.HadoopFileIO


  * Branch bronze                                   4dbce6ec15d4
  * Branch main                                     2e1cfa82b035
  * Branch silver                                   3573a469b45a
    Tag    bronze-taxi_zone_lookup-2026             07b7e396acab
    Tag    bronze-yellow_taxi-2019                  9a92d7b4e261
    Tag    bronze-yellow_taxi-2020                  6a5f2b318fc5
    Tag    bronze-yellow_taxi-2022                  25c2c8dd385f
    Tag    bronze-yellow_taxi-2023                  ce97a1dd7adb
    Tag    bronze-yellow_taxi-2025                  6491fc847bdf
    Tag    review/silver-yellow_taxi-2025-02        79aae5e5bc28
    Tag    review/silver-yellow_taxi-2025-03        604cbce59bcf
    Tag    silver-taxi_zone_lookup-2026             3573a469b45a
    Tag    silver-yellow_taxi-2019                  463fcffc30d5
    Tag    silver-yellow_taxi-2020                  2d3bf022343c
    Tag    silver-yellow_taxi-2022                  45f474018ff1
    Tag    silver-yellow_


  Branches: 3  |  Tags: 14


In [3]:
# Switch to silver branch for the rest of the notebook
nessie.set_spark_branch(spark, "silver")

spark.sql("SHOW NAMESPACES IN nessie").show(truncate=False)
spark.sql("SHOW TABLES IN nessie.silver").show(truncate=False)

26/04/19 17:50:12 INFO CatalogUtil: Loading custom FileIO implementation: org.apache.iceberg.hadoop.HadoopFileIO


+---------+
|namespace|
+---------+
|silver   |
+---------+

+---------+----------------+-----------+
|namespace|tableName       |isTemporary|
+---------+----------------+-----------+
|silver   |taxi_zone_lookup|false      |
|silver   |yellow_taxi     |false      |
+---------+----------------+-----------+



## 2. Yellow Taxi — Propiedades y esquema

In [4]:
spark.sql("DESCRIBE EXTENDED nessie.silver.yellow_taxi").show(250, truncate=False)

26/04/19 17:50:17 INFO BaseMetastoreTableOperations: Refreshing table metadata from new version: s3a://warehouse/silver/yellow_taxi_3e9198ad-6b22-4908-bb27-7390b2b3310f/metadata/00012-b0026b75-1cfb-4a27-b6f1-534f132fa880.metadata.json
26/04/19 17:50:18 INFO NessieUtil: loadTableMetadata for 'silver.yellow_taxi' from location 's3a://warehouse/silver/yellow_taxi_3e9198ad-6b22-4908-bb27-7390b2b3310f/metadata/00012-b0026b75-1cfb-4a27-b6f1-534f132fa880.metadata.json' at 'Branch{name=silver, metadata=null, hash=3573a469b45a655d0e5a59f17c4f0f9dc93464bc431b7e875d5962cf9b37ed9e}'
26/04/19 17:50:18 INFO BaseMetastoreCatalog: Table loaded by catalog: nessie.silver.yellow_taxi


+-----------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+
|col_name                     |data_type                                                                                                                                                                                                                                                                 |comment|
+-----------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+
|VendorID                     |bigint                                          

In [5]:
spark.sql("SHOW TBLPROPERTIES nessie.silver.yellow_taxi").show(truncate=False)

+------------------------------------------+----------------------------------------------------------------+
|key                                       |value                                                           |
+------------------------------------------+----------------------------------------------------------------+
|current-snapshot-id                       |5830921864270721079                                             |
|format                                    |iceberg/parquet                                                 |
|format-version                            |2                                                               |
|gc.enabled                                |false                                                           |
|nessie.commit.id                          |3573a469b45a655d0e5a59f17c4f0f9dc93464bc431b7e875d5962cf9b37ed9e|
|write.metadata.delete-after-commit.enabled|false                                                           |
|write.par

## 3. Snapshots e historial

In [6]:
spark.sql("SELECT * FROM nessie.silver.yellow_taxi.snapshots").show(truncate=False)

26/04/19 17:50:19 INFO BaseMetastoreTableOperations: Refreshing table metadata from new version: s3a://warehouse/silver/yellow_taxi_3e9198ad-6b22-4908-bb27-7390b2b3310f/metadata/00012-b0026b75-1cfb-4a27-b6f1-534f132fa880.metadata.json
26/04/19 17:50:19 INFO NessieUtil: loadTableMetadata for 'silver.yellow_taxi' from location 's3a://warehouse/silver/yellow_taxi_3e9198ad-6b22-4908-bb27-7390b2b3310f/metadata/00012-b0026b75-1cfb-4a27-b6f1-534f132fa880.metadata.json' at 'Branch{name=silver, metadata=null, hash=3573a469b45a655d0e5a59f17c4f0f9dc93464bc431b7e875d5962cf9b37ed9e}'
26/04/19 17:50:19 INFO BaseMetastoreCatalog: Table loaded by catalog: nessie.silver.yellow_taxi.snapshots
26/04/19 17:50:21 INFO SparkPartitioningAwareScan: Reporting UnknownPartitioning with 1 partition(s) for table nessie.silver.yellow_taxi.snapshots


+-----------------------+-------------------+-------------------+---------+-----------------------------------------------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id        |parent_id          |operation|manifest_list                                                                                                                                        |summary                                                    

In [7]:
spark.sql("SELECT * FROM nessie.silver.yellow_taxi.history").show(truncate=False)

26/04/19 17:50:22 INFO BaseMetastoreTableOperations: Refreshing table metadata from new version: s3a://warehouse/silver/yellow_taxi_3e9198ad-6b22-4908-bb27-7390b2b3310f/metadata/00012-b0026b75-1cfb-4a27-b6f1-534f132fa880.metadata.json
26/04/19 17:50:22 INFO NessieUtil: loadTableMetadata for 'silver.yellow_taxi' from location 's3a://warehouse/silver/yellow_taxi_3e9198ad-6b22-4908-bb27-7390b2b3310f/metadata/00012-b0026b75-1cfb-4a27-b6f1-534f132fa880.metadata.json' at 'Branch{name=silver, metadata=null, hash=3573a469b45a655d0e5a59f17c4f0f9dc93464bc431b7e875d5962cf9b37ed9e}'
26/04/19 17:50:22 INFO BaseMetastoreCatalog: Table loaded by catalog: nessie.silver.yellow_taxi.history
26/04/19 17:50:22 INFO SparkPartitioningAwareScan: Reporting UnknownPartitioning with 1 partition(s) for table nessie.silver.yellow_taxi.history


+-----------------------+-------------------+-------------------+-------------------+
|made_current_at        |snapshot_id        |parent_id          |is_current_ancestor|
+-----------------------+-------------------+-------------------+-------------------+
|2026-04-19 17:05:04.309|6745600223080055963|NULL               |true               |
|2026-04-19 17:06:35.106|4623529918104723123|6745600223080055963|true               |
|2026-04-19 17:08:14.491|5977181708450568665|4623529918104723123|true               |
|2026-04-19 17:09:43.387|4535136435007090149|5977181708450568665|true               |
|2026-04-19 17:11:08.19 |3189856653444385239|4535136435007090149|true               |
|2026-04-19 17:12:09.659|1928580236016806185|3189856653444385239|true               |
|2026-04-19 17:13:07.462|3883349896427092863|1928580236016806185|true               |
|2026-04-19 17:14:04.785|5975692737865290183|3883349896427092863|true               |
|2026-04-19 17:15:17.149|2173299420622087989|597569273

## 3b. Crecimiento de registros por tag

Cada tag `silver-yellow_taxi-{año}` captura el estado de `silver` tras procesar
todos los meses de ese año. Comparamos también con el tag equivalente en bronze.

In [8]:
import re

refs_df = spark.sql("LIST REFERENCES IN nessie")
type_col = "refType" if "refType" in refs_df.columns else "type"

silver_tags = [
    r["name"] for r in
    refs_df.filter(refs_df[type_col].ilike("tag"))
           .filter(refs_df["name"].rlike(r"^silver-yellow_taxi-\d{4}$"))
           .collect()
]
silver_tags.sort()

tag_pattern = re.compile(r"silver-yellow_taxi-(\d{4})")
rows = []
prev_silver = 0
for tag in silver_tags:
    year = int(tag_pattern.match(tag).group(1))

    nessie.set_spark_branch(spark, tag)
    silver_total = spark.table("nessie.silver.yellow_taxi").count()

    bronze_tag = f"bronze-yellow_taxi-{year}"
    try:
        nessie.set_spark_branch(spark, bronze_tag)
        bronze_total = spark.table("nessie.bronze.yellow_taxi").count()
    except Exception:
        bronze_total = None

    retention = round(silver_total / bronze_total * 100, 2) if bronze_total else None
    rows.append((tag, year, bronze_total, silver_total, silver_total - prev_silver, retention))
    prev_silver = silver_total

nessie.set_spark_branch(spark, "silver")

growth_df = spark.createDataFrame(
    rows,
    ["tag", "year", "bronze_records", "silver_records", "records_added", "retention_pct"]
)
growth_df.show(truncate=False)

26/04/19 17:50:23 INFO CatalogUtil: Loading custom FileIO implementation: org.apache.iceberg.hadoop.HadoopFileIO
26/04/19 17:50:23 INFO BaseMetastoreTableOperations: Refreshing table metadata from new version: s3a://warehouse/silver/yellow_taxi_3e9198ad-6b22-4908-bb27-7390b2b3310f/metadata/00002-b4e0b5d2-edb4-4075-b8cf-e3aa977eefe6.metadata.json
26/04/19 17:50:23 INFO NessieUtil: loadTableMetadata for 'silver.yellow_taxi' from location 's3a://warehouse/silver/yellow_taxi_3e9198ad-6b22-4908-bb27-7390b2b3310f/metadata/00002-b4e0b5d2-edb4-4075-b8cf-e3aa977eefe6.metadata.json' at 'Tag{name=silver-yellow_taxi-2019, metadata=null, hash=463fcffc30d5c29388b8dc6836803236f788dc2219efd085dfb368e70da54496}'
26/04/19 17:50:23 INFO BaseMetastoreCatalog: Table loaded by catalog: nessie.silver.yellow_taxi
26/04/19 17:50:24 INFO SnapshotScan: Scanning table nessie.silver.yellow_taxi snapshot 5977181708450568665 created at 2026-04-19T17:08:14.491+00:00 with filter true
26/04/19 17:50:24 INFO BaseDistrib

+-----------------------+----+--------------+--------------+-------------+-------------+
|tag                    |year|bronze_records|silver_records|records_added|retention_pct|
+-----------------------+----+--------------+--------------+-------------+-------------+
|silver-yellow_taxi-2019|2019|22612607      |21926817      |21926817     |96.97        |
|silver-yellow_taxi-2020|2020|38324669      |37125328      |15198511     |96.87        |
|silver-yellow_taxi-2022|2022|47395913      |45665180      |8539852      |96.35        |
|silver-yellow_taxi-2023|2023|56780400      |54573946      |8908766      |96.11        |
|silver-yellow_taxi-2025|2025|67978426      |57423106      |2849160      |84.47        |
+-----------------------+----+--------------+--------------+-------------+-------------+



## 4. Particionado y ficheros

In [9]:
spark.sql("SELECT * FROM nessie.silver.yellow_taxi.partitions").show(truncate=False)

26/04/19 17:50:37 INFO BaseMetastoreTableOperations: Refreshing table metadata from new version: s3a://warehouse/silver/yellow_taxi_3e9198ad-6b22-4908-bb27-7390b2b3310f/metadata/00012-b0026b75-1cfb-4a27-b6f1-534f132fa880.metadata.json
26/04/19 17:50:37 INFO NessieUtil: loadTableMetadata for 'silver.yellow_taxi' from location 's3a://warehouse/silver/yellow_taxi_3e9198ad-6b22-4908-bb27-7390b2b3310f/metadata/00012-b0026b75-1cfb-4a27-b6f1-534f132fa880.metadata.json' at 'Branch{name=silver, metadata=null, hash=3573a469b45a655d0e5a59f17c4f0f9dc93464bc431b7e875d5962cf9b37ed9e}'
26/04/19 17:50:37 INFO BaseMetastoreCatalog: Table loaded by catalog: nessie.silver.yellow_taxi.partitions
26/04/19 17:50:37 INFO BaseMetastoreTableOperations: Refreshing table metadata from new version: s3a://warehouse/silver/yellow_taxi_3e9198ad-6b22-4908-bb27-7390b2b3310f/metadata/00012-b0026b75-1cfb-4a27-b6f1-534f132fa880.metadata.json
26/04/19 17:50:37 INFO NessieUtil: loadTableMetadata for 'silver.yellow_taxi' fr

+---------+-------+------------+----------+-----------------------------+----------------------------+--------------------------+----------------------------+--------------------------+-----------------------+------------------------+
|partition|spec_id|record_count|file_count|total_data_file_size_in_bytes|position_delete_record_count|position_delete_file_count|equality_delete_record_count|equality_delete_file_count|last_updated_at        |last_updated_snapshot_id|
+---------+-------+------------+----------+-----------------------------+----------------------------+--------------------------+----------------------------+--------------------------+-----------------------+------------------------+
|{600}    |0      |6201962     |1         |94212089                     |0                           |0                         |0                           |0                         |2026-04-19 17:09:43.387|4535136435007090149     |
|{601}    |0      |6103376     |1         |92685401         

In [10]:
from pyspark.sql.functions import count, sum as spark_sum, round as spark_round

files_df = spark.sql("SELECT * FROM nessie.silver.yellow_taxi.files")
files_df.select(
    "content", "file_path", "file_format", "record_count", "file_size_in_bytes"
).show(100, truncate=80)

files_df.select(
    count("*").alias("total_files"),
    spark_round(spark_sum("file_size_in_bytes") / 1024 / 1024, 2).alias("total_size_mb"),
    spark_sum("record_count").alias("total_records"),
).show()

26/04/19 17:50:38 INFO BaseMetastoreTableOperations: Refreshing table metadata from new version: s3a://warehouse/silver/yellow_taxi_3e9198ad-6b22-4908-bb27-7390b2b3310f/metadata/00012-b0026b75-1cfb-4a27-b6f1-534f132fa880.metadata.json
26/04/19 17:50:38 INFO NessieUtil: loadTableMetadata for 'silver.yellow_taxi' from location 's3a://warehouse/silver/yellow_taxi_3e9198ad-6b22-4908-bb27-7390b2b3310f/metadata/00012-b0026b75-1cfb-4a27-b6f1-534f132fa880.metadata.json' at 'Branch{name=silver, metadata=null, hash=3573a469b45a655d0e5a59f17c4f0f9dc93464bc431b7e875d5962cf9b37ed9e}'
26/04/19 17:50:38 INFO BaseMetastoreCatalog: Table loaded by catalog: nessie.silver.yellow_taxi.files
26/04/19 17:50:38 INFO SnapshotScan: Scanning table nessie.silver.yellow_taxi snapshot 5830921864270721079 created at 2026-04-19T17:19:17.176+00:00 with filter true
26/04/19 17:50:38 INFO SparkPartitioningAwareScan: Reporting UnknownPartitioning with 4 partition(s) for table nessie.silver.yellow_taxi.files


+-------+--------------------------------------------------------------------------------+-----------+------------+------------------+
|content|                                                                       file_path|file_format|record_count|file_size_in_bytes|
+-------+--------------------------------------------------------------------------------+-----------+------------+------------------+
|      0|s3a://warehouse/silver/yellow_taxi_3e9198ad-6b22-4908-bb27-7390b2b3310f/data/...|    PARQUET|     2849160|          44214317|
|      0|s3a://warehouse/silver/yellow_taxi_3e9198ad-6b22-4908-bb27-7390b2b3310f/data/...|    PARQUET|     3226937|          51399336|
|      0|s3a://warehouse/silver/yellow_taxi_3e9198ad-6b22-4908-bb27-7390b2b3310f/data/...|    PARQUET|     2764185|          43299992|
|      0|s3a://warehouse/silver/yellow_taxi_3e9198ad-6b22-4908-bb27-7390b2b3310f/data/...|    PARQUET|     2917644|          45908820|
|      0|s3a://warehouse/silver/yellow_taxi_3e9198ad-6b

26/04/19 17:50:41 INFO SnapshotScan: Scanning table nessie.silver.yellow_taxi snapshot 5830921864270721079 created at 2026-04-19T17:19:17.176+00:00 with filter true
26/04/19 17:50:41 INFO SparkPartitioningAwareScan: Reporting UnknownPartitioning with 4 partition(s) for table nessie.silver.yellow_taxi.files


+-----------+-------------+-------------+
|total_files|total_size_mb|total_records|
+-----------+-------------+-------------+
|         13|       840.84|     57423106|
+-----------+-------------+-------------+



## 5. Calidad de datos — visión general

In [11]:
df = spark.table("nessie.silver.yellow_taxi")
print(f"Total records: {df.count():,}")
print(f"Columns: {len(df.columns)}")
df.printSchema()

26/04/19 17:50:42 INFO SnapshotScan: Scanning table nessie.silver.yellow_taxi snapshot 5830921864270721079 created at 2026-04-19T17:19:17.176+00:00 with filter true
26/04/19 17:50:42 INFO BaseDistributedDataScan: Planning file tasks locally for table nessie.silver.yellow_taxi


Total records: 57,423,106
Columns: 20
root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [12]:
# Registros por año-mes (valida el particionado y la cobertura temporal)
from pyspark.sql.functions import year, month

df.groupBy(
    year("tpep_pickup_datetime").alias("year"),
    month("tpep_pickup_datetime").alias("month")
).count() \
 .orderBy("year", "month") \
 .show(50)

26/04/19 17:50:43 INFO SnapshotScan: Scanning table nessie.silver.yellow_taxi snapshot 5830921864270721079 created at 2026-04-19T17:19:17.176+00:00 with filter true
26/04/19 17:50:43 INFO BaseDistributedDataScan: Planning file tasks locally for table nessie.silver.yellow_taxi
26/04/19 17:50:43 INFO SparkPartitioningAwareScan: Reporting UnknownPartitioning with 8 partition(s) for table nessie.silver.yellow_taxi


+----+-----+-------+
|year|month|  count|
+----+-----+-------+
|2019|    1|7460359|
|2019|    2|6833264|
|2019|    3|7633194|
|2020|    1|6201962|
|2020|    2|6103376|
|2020|    3|2893173|
|2022|    1|2326489|
|2022|    2|2798864|
|2022|    3|3414499|
|2023|    1|2917644|
|2023|    2|2764185|
|2023|    3|3226937|
|2025|    1|2849160|
+----+-----+-------+



In [13]:
# Nulos por columna — silver no debería tener nulos en campos críticos
from pyspark.sql.functions import col as c, sum as spark_sum

null_counts = df.select([
    spark_sum(c(col).isNull().cast("int")).alias(col)
    for col in df.columns
])
null_counts.show(vertical=True)

26/04/19 17:50:54 INFO SnapshotScan: Scanning table nessie.silver.yellow_taxi snapshot 5830921864270721079 created at 2026-04-19T17:19:17.176+00:00 with filter true
26/04/19 17:50:54 INFO BaseDistributedDataScan: Planning file tasks locally for table nessie.silver.yellow_taxi
26/04/19 17:50:55 INFO SparkPartitioningAwareScan: Reporting UnknownPartitioning with 8 partition(s) for table nessie.silver.yellow_taxi


-RECORD 0-------------------------
 VendorID              | 0        
 tpep_pickup_datetime  | 0        
 tpep_dropoff_datetime | 0        
 passenger_count       | 0        
 trip_distance         | 0        
 RatecodeID            | 0        
 store_and_fwd_flag    | 0        
 PULocationID          | 0        
 DOLocationID          | 0        
 payment_type          | 0        
 fare_amount           | 0        
 extra                 | 0        
 mta_tax               | 0        
 tip_amount            | 0        
 tolls_amount          | 0        
 improvement_surcharge | 0        
 total_amount          | 0        
 congestion_surcharge  | 4707240  
 airport_fee           | 37125328 
 cbd_congestion_fee    | 54573946 



In [14]:
# Estadísticas descriptivas
df.select(
    "trip_distance", "fare_amount", "tip_amount",
    "tolls_amount", "total_amount", "passenger_count"
).summary("count", "min", "max", "mean", "stddev").show()

26/04/19 17:51:17 INFO SnapshotScan: Scanning table nessie.silver.yellow_taxi snapshot 5830921864270721079 created at 2026-04-19T17:19:17.176+00:00 with filter true
26/04/19 17:51:17 INFO BaseDistributedDataScan: Planning file tasks locally for table nessie.silver.yellow_taxi
26/04/19 17:51:17 INFO SparkPartitioningAwareScan: Reporting UnknownPartitioning with 8 partition(s) for table nessie.silver.yellow_taxi
26/04/19 17:51:17 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+------------------+------------------+------------------+-------------------+------------------+------------------+
|summary|     trip_distance|       fare_amount|        tip_amount|       tolls_amount|      total_amount|   passenger_count|
+-------+------------------+------------------+------------------+-------------------+------------------+------------------+
|  count|          57423106|          57423106|          57423106|           57423106|          57423106|          57423106|
|    min|               0.0|               0.0|               0.0|                0.0|               0.0|               1.0|
|    max|          62359.52|          943274.8|         141492.02|             3288.0|        1084772.17|               6.0|
|   mean|3.0244733053972817|14.032787563074745|2.4666192333839834|0.39038326575299515|20.370920402435495|1.5076177523382313|
| stddev| 17.90943567829969|233.28387288652902|18.917940338737726| 1.8303189346291913| 277.3682734043703|1.0996552888594409|


## 6. Validación de las reglas de limpieza

Cada check debe devolver **0 registros**. Si alguno devuelve filas, la regla de limpieza falló.

In [15]:
from pyspark.sql.functions import col

# Timestamps nulos o inválidos
print("Null pickup:",  df.filter(col("tpep_pickup_datetime").isNull()).count())
print("Null dropoff:", df.filter(col("tpep_dropoff_datetime").isNull()).count())
print("Pickup >= dropoff:",
      df.filter(col("tpep_pickup_datetime") >= col("tpep_dropoff_datetime")).count())

26/04/19 17:51:28 INFO SparkScanBuilder: Evaluating completely on Iceberg side: tpep_pickup_datetime IS NULL
26/04/19 17:51:28 INFO SnapshotScan: Scanning table nessie.silver.yellow_taxi snapshot 5830921864270721079 created at 2026-04-19T17:19:17.176+00:00 with filter tpep_pickup_datetime IS NULL
26/04/19 17:51:28 INFO BaseDistributedDataScan: Planning file tasks locally for table nessie.silver.yellow_taxi


Null pickup: 0


26/04/19 17:51:29 INFO SnapshotScan: Scanning table nessie.silver.yellow_taxi snapshot 5830921864270721079 created at 2026-04-19T17:19:17.176+00:00 with filter tpep_dropoff_datetime IS NULL
26/04/19 17:51:29 INFO BaseDistributedDataScan: Planning file tasks locally for table nessie.silver.yellow_taxi
26/04/19 17:51:29 INFO SparkPartitioningAwareScan: Reporting UnknownPartitioning with 0 partition(s) for table nessie.silver.yellow_taxi


Null dropoff: 0


26/04/19 17:51:29 INFO SparkScanBuilder: Evaluating completely on Iceberg side: tpep_pickup_datetime IS NOT NULL
26/04/19 17:51:29 INFO SnapshotScan: Scanning table nessie.silver.yellow_taxi snapshot 5830921864270721079 created at 2026-04-19T17:19:17.176+00:00 with filter (tpep_pickup_datetime IS NOT NULL AND tpep_dropoff_datetime IS NOT NULL)
26/04/19 17:51:29 INFO BaseDistributedDataScan: Planning file tasks locally for table nessie.silver.yellow_taxi
26/04/19 17:51:30 INFO SparkPartitioningAwareScan: Reporting UnknownPartitioning with 8 partition(s) for table nessie.silver.yellow_taxi


Pickup >= dropoff: 0


In [16]:
# Año de pickup no coincide con el año esperado del fichero
from pyspark.sql.functions import year as spark_year

VALID_YEARS = [2019, 2020, 2022, 2023, 2025]
wrong_year = df.filter(~spark_year(col("tpep_pickup_datetime")).isin(VALID_YEARS))
print(f"Wrong-year rows: {wrong_year.count()}")
if wrong_year.count() > 0:
    wrong_year.groupBy(spark_year("tpep_pickup_datetime").alias("year")).count().show()

26/04/19 17:51:35 INFO SnapshotScan: Scanning table nessie.silver.yellow_taxi snapshot 5830921864270721079 created at 2026-04-19T17:19:17.176+00:00 with filter true
26/04/19 17:51:35 INFO BaseDistributedDataScan: Planning file tasks locally for table nessie.silver.yellow_taxi
26/04/19 17:51:35 INFO SparkPartitioningAwareScan: Reporting UnknownPartitioning with 8 partition(s) for table nessie.silver.yellow_taxi
26/04/19 17:51:41 INFO SnapshotScan: Scanning table nessie.silver.yellow_taxi snapshot 5830921864270721079 created at 2026-04-19T17:19:17.176+00:00 with filter true
26/04/19 17:51:41 INFO BaseDistributedDataScan: Planning file tasks locally for table nessie.silver.yellow_taxi


Wrong-year rows: 0


26/04/19 17:51:41 INFO SparkPartitioningAwareScan: Reporting UnknownPartitioning with 8 partition(s) for table nessie.silver.yellow_taxi


In [17]:
# Valores negativos en campos numéricos
checks = {
    "trip_distance < 0":  col("trip_distance")  < 0,
    "fare_amount < 0":    col("fare_amount")    < 0,
    "total_amount < 0":   col("total_amount")   < 0,
    "tip_amount < 0":     col("tip_amount")     < 0,
    "tolls_amount < 0":   col("tolls_amount")   < 0,
}
for label, cond in checks.items():
    print(f"{label}: {df.filter(cond).count()}")

26/04/19 17:51:47 INFO SnapshotScan: Scanning table nessie.silver.yellow_taxi snapshot 5830921864270721079 created at 2026-04-19T17:19:17.176+00:00 with filter (trip_distance IS NOT NULL AND trip_distance < (1-digit-float))
26/04/19 17:51:47 INFO BaseDistributedDataScan: Planning file tasks locally for table nessie.silver.yellow_taxi
26/04/19 17:51:47 INFO SparkPartitioningAwareScan: Reporting UnknownPartitioning with 0 partition(s) for table nessie.silver.yellow_taxi


trip_distance < 0: 0


26/04/19 17:51:48 INFO SnapshotScan: Scanning table nessie.silver.yellow_taxi snapshot 5830921864270721079 created at 2026-04-19T17:19:17.176+00:00 with filter (fare_amount IS NOT NULL AND fare_amount < (1-digit-float))
26/04/19 17:51:48 INFO BaseDistributedDataScan: Planning file tasks locally for table nessie.silver.yellow_taxi
26/04/19 17:51:48 INFO SparkPartitioningAwareScan: Reporting UnknownPartitioning with 0 partition(s) for table nessie.silver.yellow_taxi


fare_amount < 0: 0


26/04/19 17:51:49 INFO SnapshotScan: Scanning table nessie.silver.yellow_taxi snapshot 5830921864270721079 created at 2026-04-19T17:19:17.176+00:00 with filter (total_amount IS NOT NULL AND total_amount < (1-digit-float))
26/04/19 17:51:49 INFO BaseDistributedDataScan: Planning file tasks locally for table nessie.silver.yellow_taxi
26/04/19 17:51:49 INFO SparkPartitioningAwareScan: Reporting UnknownPartitioning with 0 partition(s) for table nessie.silver.yellow_taxi


total_amount < 0: 0


26/04/19 17:51:50 INFO SnapshotScan: Scanning table nessie.silver.yellow_taxi snapshot 5830921864270721079 created at 2026-04-19T17:19:17.176+00:00 with filter (tip_amount IS NOT NULL AND tip_amount < (1-digit-float))
26/04/19 17:51:50 INFO BaseDistributedDataScan: Planning file tasks locally for table nessie.silver.yellow_taxi
26/04/19 17:51:50 INFO SparkPartitioningAwareScan: Reporting UnknownPartitioning with 0 partition(s) for table nessie.silver.yellow_taxi


tip_amount < 0: 0


26/04/19 17:51:50 INFO SnapshotScan: Scanning table nessie.silver.yellow_taxi snapshot 5830921864270721079 created at 2026-04-19T17:19:17.176+00:00 with filter (tolls_amount IS NOT NULL AND tolls_amount < (1-digit-float))
26/04/19 17:51:50 INFO BaseDistributedDataScan: Planning file tasks locally for table nessie.silver.yellow_taxi
26/04/19 17:51:50 INFO SparkPartitioningAwareScan: Reporting UnknownPartitioning with 0 partition(s) for table nessie.silver.yellow_taxi


tolls_amount < 0: 0


In [18]:
# Location IDs fuera del rango válido (1–265)
print("Invalid PULocationID:",
      df.filter(~col("PULocationID").between(1, 265)).count())
print("Invalid DOLocationID:",
      df.filter(~col("DOLocationID").between(1, 265)).count())

26/04/19 17:51:51 INFO SnapshotScan: Scanning table nessie.silver.yellow_taxi snapshot 5830921864270721079 created at 2026-04-19T17:19:17.176+00:00 with filter (PULocationID < (1-digit-int) OR PULocationID > (3-digit-int))
26/04/19 17:51:51 INFO BaseDistributedDataScan: Planning file tasks locally for table nessie.silver.yellow_taxi
26/04/19 17:51:51 INFO SparkPartitioningAwareScan: Reporting UnknownPartitioning with 0 partition(s) for table nessie.silver.yellow_taxi


Invalid PULocationID: 0


26/04/19 17:51:52 INFO SnapshotScan: Scanning table nessie.silver.yellow_taxi snapshot 5830921864270721079 created at 2026-04-19T17:19:17.176+00:00 with filter (DOLocationID < (1-digit-int) OR DOLocationID > (3-digit-int))
26/04/19 17:51:52 INFO BaseDistributedDataScan: Planning file tasks locally for table nessie.silver.yellow_taxi


Invalid DOLocationID: 0


26/04/19 17:51:52 INFO SparkPartitioningAwareScan: Reporting UnknownPartitioning with 0 partition(s) for table nessie.silver.yellow_taxi


In [19]:
# Valores categóricos fuera del diccionario TLC
VALID_VENDOR_IDS    = [1, 2, 6, 7]
VALID_RATE_CODES    = [1, 2, 3, 4, 5, 6, 99]
VALID_PAYMENT_TYPES = [0, 1, 2, 3, 4, 5, 6]

print("Invalid VendorID:",
      df.filter(~col("VendorID").isin(VALID_VENDOR_IDS)).count())
print("Invalid RatecodeID:",
      df.filter(~col("RatecodeID").isin(VALID_RATE_CODES)).count())
print("Invalid payment_type:",
      df.filter(~col("payment_type").isin(VALID_PAYMENT_TYPES)).count())
print("Invalid store_and_fwd_flag:",
      df.filter(~col("store_and_fwd_flag").isin("Y", "N")).count())
print("Invalid passenger_count:",
      df.filter(~col("passenger_count").between(1, 6)).count())

26/04/19 17:51:52 INFO SnapshotScan: Scanning table nessie.silver.yellow_taxi snapshot 5830921864270721079 created at 2026-04-19T17:19:17.176+00:00 with filter (VendorID IS NOT NULL AND VendorID NOT IN ((1-digit-int), (1-digit-int), (1-digit-int), (1-digit-int)))
26/04/19 17:51:52 INFO BaseDistributedDataScan: Planning file tasks locally for table nessie.silver.yellow_taxi
26/04/19 17:51:53 INFO SparkPartitioningAwareScan: Reporting UnknownPartitioning with 8 partition(s) for table nessie.silver.yellow_taxi


Invalid VendorID: 0


26/04/19 17:51:54 INFO SnapshotScan: Scanning table nessie.silver.yellow_taxi snapshot 5830921864270721079 created at 2026-04-19T17:19:17.176+00:00 with filter (RatecodeID IS NOT NULL AND RatecodeID NOT IN ((1-digit-float), (1-digit-float), (1-digit-float), (1-digit-float), (1-digit-float), (1-digit-float), (2-digit-float)))
26/04/19 17:51:54 INFO BaseDistributedDataScan: Planning file tasks locally for table nessie.silver.yellow_taxi
26/04/19 17:51:54 INFO SparkPartitioningAwareScan: Reporting UnknownPartitioning with 8 partition(s) for table nessie.silver.yellow_taxi


Invalid RatecodeID: 0


26/04/19 17:51:55 INFO SnapshotScan: Scanning table nessie.silver.yellow_taxi snapshot 5830921864270721079 created at 2026-04-19T17:19:17.176+00:00 with filter (payment_type IS NOT NULL AND payment_type NOT IN ((1-digit-int), (1-digit-int), (1-digit-int), (1-digit-int), (1-digit-int), (1-digit-int), (1-digit-int)))
26/04/19 17:51:55 INFO BaseDistributedDataScan: Planning file tasks locally for table nessie.silver.yellow_taxi
26/04/19 17:51:55 INFO SparkPartitioningAwareScan: Reporting UnknownPartitioning with 8 partition(s) for table nessie.silver.yellow_taxi


...

26/04/19 17:51:56 INFO SnapshotScan: Scanning table nessie.silver.yellow_taxi snapshot 5830921864270721079 created at 2026-04-19T17:19:17.176+00:00 with filter (store_and_fwd_flag IS NOT NULL AND store_and_fwd_flag NOT IN ((hash-5d29f651), (hash-662e0bb1)))
26/04/19 17:51:56 INFO BaseDistributedDataScan: Planning file tasks locally for table nessie.silver.yellow_taxi
26/04/19 17:51:57 INFO SparkPartitioningAwareScan: Reporting UnknownPartitioning with 8 partition(s) for table nessie.silver.yellow_taxi


Invalid store_and_fwd_flag: 0


26/04/19 17:51:57 INFO SnapshotScan: Scanning table nessie.silver.yellow_taxi snapshot 5830921864270721079 created at 2026-04-19T17:19:17.176+00:00 with filter (passenger_count < (1-digit-float) OR passenger_count > (1-digit-float))
26/04/19 17:51:57 INFO BaseDistributedDataScan: Planning file tasks locally for table nessie.silver.yellow_taxi


Invalid passenger_count: 0


26/04/19 17:51:58 INFO SparkPartitioningAwareScan: Reporting UnknownPartitioning with 0 partition(s) for table nessie.silver.yellow_taxi


In [20]:
# Distribución de los campos categóricos
for field in ["VendorID", "RatecodeID", "payment_type", "store_and_fwd_flag", "passenger_count"]:
    print(f"--- {field} ---")
    df.groupBy(field).count().orderBy(field).show()

--- VendorID ---


26/04/19 17:51:58 INFO SparkScanBuilder: Skipping aggregate pushdown: group by aggregation push down is not supported
26/04/19 17:51:58 INFO SnapshotScan: Scanning table nessie.silver.yellow_taxi snapshot 5830921864270721079 created at 2026-04-19T17:19:17.176+00:00 with filter true
26/04/19 17:51:58 INFO BaseDistributedDataScan: Planning file tasks locally for table nessie.silver.yellow_taxi
26/04/19 17:51:58 INFO SparkPartitioningAwareScan: Reporting UnknownPartitioning with 8 partition(s) for table nessie.silver.yellow_taxi
26/04/19 17:52:01 INFO SparkScanBuilder: Skipping aggregate pushdown: group by aggregation push down is not supported
26/04/19 17:52:01 INFO SnapshotScan: Scanning table nessie.silver.yellow_taxi snapshot 5830921864270721079 created at 2026-04-19T17:19:17.176+00:00 with filter true
26/04/19 17:52:01 INFO BaseDistributedDataScan: Planning file tasks locally for table nessie.silver.yellow_taxi


+--------+--------+
|VendorID|   count|
+--------+--------+
|       1|18381557|
|       2|39041549|
+--------+--------+

--- RatecodeID ---


26/04/19 17:52:02 INFO SparkPartitioningAwareScan: Reporting UnknownPartitioning with 8 partition(s) for table nessie.silver.yellow_taxi
26/04/19 17:52:05 INFO SparkScanBuilder: Skipping aggregate pushdown: group by aggregation push down is not supported
26/04/19 17:52:05 INFO SnapshotScan: Scanning table nessie.silver.yellow_taxi snapshot 5830921864270721079 created at 2026-04-19T17:19:17.176+00:00 with filter true
26/04/19 17:52:05 INFO BaseDistributedDataScan: Planning file tasks locally for table nessie.silver.yellow_taxi


+----------+--------+
|RatecodeID|   count|
+----------+--------+
|       1.0|55236091|
|       2.0| 1529295|
|       3.0|  120714|
|       4.0|   57226|
|       5.0|  359635|
|       6.0|     270|
|      99.0|  119875|
+----------+--------+

--- payment_type ---


...

+------------+--------+
|payment_type|   count|
+------------+--------+
|           1|43909598|
|           2|13136865|
|           3|  224501|
|           4|  152136|
|           5|       6|
+------------+--------+

--- store_and_fwd_flag ---


26/04/19 17:52:08 INFO SparkPartitioningAwareScan: Reporting UnknownPartitioning with 8 partition(s) for table nessie.silver.yellow_taxi
26/04/19 17:52:11 INFO SparkScanBuilder: Skipping aggregate pushdown: group by aggregation push down is not supported
26/04/19 17:52:11 INFO SnapshotScan: Scanning table nessie.silver.yellow_taxi snapshot 5830921864270721079 created at 2026-04-19T17:19:17.176+00:00 with filter true
26/04/19 17:52:11 INFO BaseDistributedDataScan: Planning file tasks locally for table nessie.silver.yellow_taxi


+------------------+--------+
|store_and_fwd_flag|   count|
+------------------+--------+
|                 N|56959221|
|                 Y|  463885|
+------------------+--------+

--- passenger_count ---


26/04/19 17:52:12 INFO SparkPartitioningAwareScan: Reporting UnknownPartitioning with 8 partition(s) for table nessie.silver.yellow_taxi


+---------------+--------+
|passenger_count|   count|
+---------------+--------+
|            1.0|42582255|
|            2.0| 8592285|
|            3.0| 2250059|
|            4.0| 1051657|
|            5.0| 1832636|
|            6.0| 1114214|
+---------------+--------+



## 7. Comparativa Bronze vs Silver por mes

Muestra cuántas filas se eliminaron en la limpieza, mes a mes.

In [21]:
from pyspark.sql.functions import year, month, count, round as spark_round

def monthly_counts(table: str, branch: str):
    nessie.set_spark_branch(spark, branch)
    return (
        spark.table(table)
        .groupBy(
            year("tpep_pickup_datetime").alias("year"),
            month("tpep_pickup_datetime").alias("month")
        )
        .agg(count("*").alias(f"{branch}_count"))
    )

bronze_counts = monthly_counts("nessie.bronze.yellow_taxi", "bronze")
silver_counts = monthly_counts("nessie.silver.yellow_taxi", "silver")

comparison = bronze_counts.join(silver_counts, on=["year", "month"], how="outer") \
    .withColumn("dropped", col("bronze_count") - col("silver_count")) \
    .withColumn("retention_pct",
                spark_round(col("silver_count") / col("bronze_count") * 100, 2)) \
    .orderBy("year", "month")

comparison.show(50)

26/04/19 17:52:15 INFO CatalogUtil: Loading custom FileIO implementation: org.apache.iceberg.hadoop.HadoopFileIO
26/04/19 17:52:15 INFO BaseMetastoreTableOperations: Refreshing table metadata from new version: s3a://warehouse/bronze/yellow_taxi_50751f92-b6b0-4a48-aecc-8b3e475433a5/metadata/00015-b0a8efb0-a5da-44f9-9c05-70af3c6c76f2.metadata.json
26/04/19 17:52:15 INFO NessieUtil: loadTableMetadata for 'bronze.yellow_taxi' from location 's3a://warehouse/bronze/yellow_taxi_50751f92-b6b0-4a48-aecc-8b3e475433a5/metadata/00015-b0a8efb0-a5da-44f9-9c05-70af3c6c76f2.metadata.json' at 'Branch{name=bronze, metadata=null, hash=4dbce6ec15d4634d4c5699c188b0c701b5f5ab9790f1a2a8c3f3b061c3367773}'
26/04/19 17:52:15 INFO BaseMetastoreCatalog: Table loaded by catalog: nessie.bronze.yellow_taxi
26/04/19 17:52:15 INFO CatalogUtil: Loading custom FileIO implementation: org.apache.iceberg.hadoop.HadoopFileIO
26/04/19 17:52:15 INFO BaseMetastoreTableOperations: Refreshing table metadata from new version: s3a

+----+-----+------------+------------+-------+-------------+
|year|month|bronze_count|silver_count|dropped|retention_pct|
+----+-----+------------+------------+-------+-------------+
|2001|    1|           3|        NULL|   NULL|         NULL|
|2001|    2|           1|        NULL|   NULL|         NULL|
|2002|   12|           3|        NULL|   NULL|         NULL|
|2003|    1|           8|        NULL|   NULL|         NULL|
|2007|   12|           1|        NULL|   NULL|         NULL|
|2008|   12|          92|        NULL|   NULL|         NULL|
|2009|    1|         227|        NULL|   NULL|         NULL|
|2012|    2|           1|        NULL|   NULL|         NULL|
|2014|   11|           1|        NULL|   NULL|         NULL|
|2018|   11|          11|        NULL|   NULL|         NULL|
|2018|   12|         355|        NULL|   NULL|         NULL|
|2019|    1|     7696390|     7460359| 236031|        96.93|
|2019|    2|     7049178|     6833264| 215914|        96.94|
|2019|    3|     7866239

In [22]:
# Totales agregados
from pyspark.sql.functions import sum as spark_sum

nessie.set_spark_branch(spark, "silver")
comparison.select(
    spark_sum("bronze_count").alias("total_bronze"),
    spark_sum("silver_count").alias("total_silver"),
    spark_sum("dropped").alias("total_dropped"),
    spark_round(
        spark_sum("silver_count") / spark_sum("bronze_count") * 100, 2
    ).alias("overall_retention_pct")
).show()

26/04/19 17:52:32 INFO CatalogUtil: Loading custom FileIO implementation: org.apache.iceberg.hadoop.HadoopFileIO
26/04/19 17:52:32 INFO SnapshotScan: Scanning table nessie.bronze.yellow_taxi snapshot 319035089640276877 created at 2026-04-12T20:00:31.249+00:00 with filter true
26/04/19 17:52:32 INFO BaseDistributedDataScan: Planning file tasks locally for table nessie.bronze.yellow_taxi
26/04/19 17:52:33 INFO SparkPartitioningAwareScan: Reporting UnknownPartitioning with 12 partition(s) for table nessie.bronze.yellow_taxi
26/04/19 17:52:33 INFO SnapshotScan: Scanning table nessie.silver.yellow_taxi snapshot 5830921864270721079 created at 2026-04-19T17:19:17.176+00:00 with filter true
26/04/19 17:52:33 INFO BaseDistributedDataScan: Planning file tasks locally for table nessie.silver.yellow_taxi
26/04/19 17:52:33 INFO SparkPartitioningAwareScan: Reporting UnknownPartitioning with 8 partition(s) for table nessie.silver.yellow_taxi


+------------+------------+-------------+---------------------+
|total_bronze|total_silver|total_dropped|overall_retention_pct|
+------------+------------+-------------+---------------------+
|    67978426|    57423106|      2831204|                84.47|
+------------+------------+-------------+---------------------+



## 8. Muestra de datos

In [23]:
nessie.set_spark_branch(spark, "silver")
df = spark.table("nessie.silver.yellow_taxi")
df.show(5, truncate=False)

26/04/19 17:52:49 INFO CatalogUtil: Loading custom FileIO implementation: org.apache.iceberg.hadoop.HadoopFileIO
26/04/19 17:52:49 INFO BaseMetastoreTableOperations: Refreshing table metadata from new version: s3a://warehouse/silver/yellow_taxi_3e9198ad-6b22-4908-bb27-7390b2b3310f/metadata/00012-b0026b75-1cfb-4a27-b6f1-534f132fa880.metadata.json
26/04/19 17:52:49 INFO NessieUtil: loadTableMetadata for 'silver.yellow_taxi' from location 's3a://warehouse/silver/yellow_taxi_3e9198ad-6b22-4908-bb27-7390b2b3310f/metadata/00012-b0026b75-1cfb-4a27-b6f1-534f132fa880.metadata.json' at 'Branch{name=silver, metadata=null, hash=3573a469b45a655d0e5a59f17c4f0f9dc93464bc431b7e875d5962cf9b37ed9e}'
26/04/19 17:52:49 INFO BaseMetastoreCatalog: Table loaded by catalog: nessie.silver.yellow_taxi
26/04/19 17:52:49 INFO SnapshotScan: Scanning table nessie.silver.yellow_taxi snapshot 5830921864270721079 created at 2026-04-19T17:19:17.176+00:00 with filter true
26/04/19 17:52:50 INFO BaseDistributedDataScan: 

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|2       |2023-02-24 08:02:09 |2023-02-24 08:38:56  |1.0            |15.6         |3.0       |N                 |230         |1   

## 9. Taxi zone lookup — esquema y calidad


In [24]:
zones = spark.table("nessie.silver.taxi_zone_lookup")
print(f"Records: {zones.count()}")
zones.printSchema()
spark.sql("DESCRIBE EXTENDED nessie.silver.taxi_zone_lookup").show(50, truncate=False)


26/04/19 17:52:51 INFO BaseMetastoreTableOperations: Refreshing table metadata from new version: s3a://warehouse/silver/taxi_zone_lookup_91e81872-97b3-4f51-9e4b-95c03f055dcd/metadata/00000-8115f842-5fbe-455f-8fed-3dc49c48aaa7.metadata.json
26/04/19 17:52:51 INFO NessieUtil: loadTableMetadata for 'silver.taxi_zone_lookup' from location 's3a://warehouse/silver/taxi_zone_lookup_91e81872-97b3-4f51-9e4b-95c03f055dcd/metadata/00000-8115f842-5fbe-455f-8fed-3dc49c48aaa7.metadata.json' at 'Branch{name=silver, metadata=null, hash=3573a469b45a655d0e5a59f17c4f0f9dc93464bc431b7e875d5962cf9b37ed9e}'
26/04/19 17:52:51 INFO BaseMetastoreCatalog: Table loaded by catalog: nessie.silver.taxi_zone_lookup
26/04/19 17:52:51 INFO SnapshotScan: Scanning table nessie.silver.taxi_zone_lookup snapshot 2672599193303667998 created at 2026-04-19T17:25:47.429+00:00 with filter true
26/04/19 17:52:51 INFO BaseDistributedDataScan: Planning file tasks locally for table nessie.silver.taxi_zone_lookup


Records: 265
root
 |-- location_id: integer (nullable = true)
 |-- borough: string (nullable = true)
 |-- zone: string (nullable = true)
 |-- service_zone: string (nullable = true)

+-----------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------+
|col_name                     |data_type                                                                                                                                                                                                                                                                 |comment|
+-----------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [25]:
# Nulos en campos críticos y rango del location_id
from pyspark.sql.functions import col

print("Null location_id:", zones.filter(col("location_id").isNull()).count())
print("Null borough   :", zones.filter(col("borough").isNull() | (col("borough") == "")).count())
print("Null zone      :", zones.filter(col("zone").isNull() | (col("zone") == "")).count())
print("Out-of-range location_id:",
      zones.filter(~col("location_id").between(1, 265)).count())

# Duplicados por location_id (clave lógica)
dupes = (zones.groupBy("location_id").count()
         .filter(col("count") > 1))
print("Duplicate location_id:", dupes.count())
if dupes.count() > 0:
    dupes.show()


26/04/19 17:52:52 INFO SnapshotScan: Scanning table nessie.silver.taxi_zone_lookup snapshot 2672599193303667998 created at 2026-04-19T17:25:47.429+00:00 with filter location_id IS NULL
26/04/19 17:52:52 INFO BaseDistributedDataScan: Planning file tasks locally for table nessie.silver.taxi_zone_lookup
26/04/19 17:52:52 INFO SparkPartitioningAwareScan: Reporting UnknownPartitioning with 0 partition(s) for table nessie.silver.taxi_zone_lookup


Null location_id: 0


26/04/19 17:52:52 INFO SnapshotScan: Scanning table nessie.silver.taxi_zone_lookup snapshot 2672599193303667998 created at 2026-04-19T17:25:47.429+00:00 with filter (borough IS NULL OR borough = (hash-00000000))
26/04/19 17:52:52 INFO BaseDistributedDataScan: Planning file tasks locally for table nessie.silver.taxi_zone_lookup
26/04/19 17:52:52 INFO SparkPartitioningAwareScan: Reporting UnknownPartitioning with 0 partition(s) for table nessie.silver.taxi_zone_lookup


Null borough   : 0
Null zone      : 0


26/04/19 17:52:53 INFO SnapshotScan: Scanning table nessie.silver.taxi_zone_lookup snapshot 2672599193303667998 created at 2026-04-19T17:25:47.429+00:00 with filter (zone IS NULL OR zone = (hash-00000000))
26/04/19 17:52:53 INFO BaseDistributedDataScan: Planning file tasks locally for table nessie.silver.taxi_zone_lookup
26/04/19 17:52:53 INFO SparkPartitioningAwareScan: Reporting UnknownPartitioning with 0 partition(s) for table nessie.silver.taxi_zone_lookup
26/04/19 17:52:53 INFO SnapshotScan: Scanning table nessie.silver.taxi_zone_lookup snapshot 2672599193303667998 created at 2026-04-19T17:25:47.429+00:00 with filter (location_id < (1-digit-int) OR location_id > (3-digit-int))
26/04/19 17:52:53 INFO BaseDistributedDataScan: Planning file tasks locally for table nessie.silver.taxi_zone_lookup
26/04/19 17:52:53 INFO SparkPartitioningAwareScan: Reporting UnknownPartitioning with 0 partition(s) for table nessie.silver.taxi_zone_lookup


Out-of-range location_id: 0


26/04/19 17:52:54 INFO SparkScanBuilder: Skipping aggregate pushdown: group by aggregation push down is not supported
26/04/19 17:52:54 INFO SnapshotScan: Scanning table nessie.silver.taxi_zone_lookup snapshot 2672599193303667998 created at 2026-04-19T17:25:47.429+00:00 with filter true
26/04/19 17:52:54 INFO BaseDistributedDataScan: Planning file tasks locally for table nessie.silver.taxi_zone_lookup
26/04/19 17:52:54 INFO SparkPartitioningAwareScan: Reporting UnknownPartitioning with 1 partition(s) for table nessie.silver.taxi_zone_lookup


Duplicate location_id: 0


26/04/19 17:52:54 INFO SparkScanBuilder: Skipping aggregate pushdown: group by aggregation push down is not supported
26/04/19 17:52:54 INFO SnapshotScan: Scanning table nessie.silver.taxi_zone_lookup snapshot 2672599193303667998 created at 2026-04-19T17:25:47.429+00:00 with filter true
26/04/19 17:52:54 INFO BaseDistributedDataScan: Planning file tasks locally for table nessie.silver.taxi_zone_lookup
26/04/19 17:52:54 INFO SparkPartitioningAwareScan: Reporting UnknownPartitioning with 1 partition(s) for table nessie.silver.taxi_zone_lookup


In [26]:
# Distribución por borough y service_zone
print("--- borough ---")
zones.groupBy("borough").count().orderBy("borough").show(truncate=False)
print("--- service_zone ---")
zones.groupBy("service_zone").count().orderBy("service_zone").show(truncate=False)


--- borough ---


26/04/19 17:52:55 INFO SparkScanBuilder: Skipping aggregate pushdown: group by aggregation push down is not supported
26/04/19 17:52:55 INFO SnapshotScan: Scanning table nessie.silver.taxi_zone_lookup snapshot 2672599193303667998 created at 2026-04-19T17:25:47.429+00:00 with filter true
26/04/19 17:52:55 INFO BaseDistributedDataScan: Planning file tasks locally for table nessie.silver.taxi_zone_lookup
26/04/19 17:52:55 INFO SparkPartitioningAwareScan: Reporting UnknownPartitioning with 1 partition(s) for table nessie.silver.taxi_zone_lookup


+-------------+-----+
|borough      |count|
+-------------+-----+
|Bronx        |43   |
|Brooklyn     |61   |
|EWR          |1    |
|Manhattan    |69   |
|N/A          |1    |
|Queens       |69   |
|Staten Island|20   |
|Unknown      |1    |
+-------------+-----+

--- service_zone ---


26/04/19 17:52:56 INFO SparkScanBuilder: Skipping aggregate pushdown: group by aggregation push down is not supported
26/04/19 17:52:56 INFO SnapshotScan: Scanning table nessie.silver.taxi_zone_lookup snapshot 2672599193303667998 created at 2026-04-19T17:25:47.429+00:00 with filter true
26/04/19 17:52:56 INFO BaseDistributedDataScan: Planning file tasks locally for table nessie.silver.taxi_zone_lookup
26/04/19 17:52:56 INFO SparkPartitioningAwareScan: Reporting UnknownPartitioning with 1 partition(s) for table nessie.silver.taxi_zone_lookup


+------------+-----+
|service_zone|count|
+------------+-----+
|Airports    |2    |
|Boro Zone   |205  |
|EWR         |1    |
|N/A         |2    |
|Yellow Zone |55   |
+------------+-----+



In [27]:
# Comparativa Bronze vs Silver (zone lookup es pequeno, ~265 filas: comparamos conjuntos de IDs)
nessie.set_spark_branch(spark, "bronze")
bronze_ids = {row["LocationID"] for row in
              spark.table("nessie.bronze.taxi_zone_lookup").select("LocationID").collect()}
nessie.set_spark_branch(spark, "silver")
silver_ids = {row["location_id"] for row in zones.select("location_id").collect()}

b, s = len(bronze_ids), len(silver_ids)
if b:
    print(f"Bronze: {b}  Silver: {s}  Dropped: {b - s}  Retention: {s / b:.1%}")
else:
    print("Bronze empty")

missing = sorted(bronze_ids - silver_ids)
print(f"IDs in bronze but not in silver: {len(missing)}")
if missing:
    print("  " + ", ".join(str(i) for i in missing))


26/04/19 17:52:56 INFO CatalogUtil: Loading custom FileIO implementation: org.apache.iceberg.hadoop.HadoopFileIO
26/04/19 17:52:56 INFO BaseMetastoreTableOperations: Refreshing table metadata from new version: s3a://warehouse/bronze/taxi_zone_lookup_4a158c56-0ae5-41c7-b807-936027cd2e75/metadata/00001-e7156de2-b6e8-4526-9c5e-7e5a9a3d0748.metadata.json
26/04/19 17:52:56 INFO NessieUtil: loadTableMetadata for 'bronze.taxi_zone_lookup' from location 's3a://warehouse/bronze/taxi_zone_lookup_4a158c56-0ae5-41c7-b807-936027cd2e75/metadata/00001-e7156de2-b6e8-4526-9c5e-7e5a9a3d0748.metadata.json' at 'Branch{name=bronze, metadata=null, hash=4dbce6ec15d4634d4c5699c188b0c701b5f5ab9790f1a2a8c3f3b061c3367773}'
26/04/19 17:52:56 INFO BaseMetastoreCatalog: Table loaded by catalog: nessie.bronze.taxi_zone_lookup
26/04/19 17:52:56 INFO SnapshotScan: Scanning table nessie.bronze.taxi_zone_lookup snapshot 6288921910691392758 created at 2026-04-19T17:39:17.343+00:00 with filter true
26/04/19 17:52:57 INFO 

Bronze: 265  Silver: 265  Dropped: 0  Retention: 100.0%
IDs in bronze but not in silver: 0


In [28]:
# Integridad referencial: ¿todas las PULocationID/DOLocationID de silver.yellow_taxi
# existen en silver.taxi_zone_lookup?
trips = spark.table("nessie.silver.yellow_taxi")

pu_missing = (trips.select(col("PULocationID").alias("location_id")).distinct()
              .join(zones.select("location_id"), "location_id", "left_anti"))
do_missing = (trips.select(col("DOLocationID").alias("location_id")).distinct()
              .join(zones.select("location_id"), "location_id", "left_anti"))

print("PULocationID not found in zone lookup:", pu_missing.count())
pu_missing.show(truncate=False)
print("DOLocationID not found in zone lookup:", do_missing.count())
do_missing.show(truncate=False)


26/04/19 17:52:57 INFO BaseMetastoreTableOperations: Refreshing table metadata from new version: s3a://warehouse/silver/yellow_taxi_3e9198ad-6b22-4908-bb27-7390b2b3310f/metadata/00012-b0026b75-1cfb-4a27-b6f1-534f132fa880.metadata.json
26/04/19 17:52:57 INFO NessieUtil: loadTableMetadata for 'silver.yellow_taxi' from location 's3a://warehouse/silver/yellow_taxi_3e9198ad-6b22-4908-bb27-7390b2b3310f/metadata/00012-b0026b75-1cfb-4a27-b6f1-534f132fa880.metadata.json' at 'Branch{name=silver, metadata=null, hash=3573a469b45a655d0e5a59f17c4f0f9dc93464bc431b7e875d5962cf9b37ed9e}'
26/04/19 17:52:57 INFO BaseMetastoreCatalog: Table loaded by catalog: nessie.silver.yellow_taxi
26/04/19 17:52:57 INFO SnapshotScan: Scanning table nessie.silver.taxi_zone_lookup snapshot 2672599193303667998 created at 2026-04-19T17:25:47.429+00:00 with filter true
26/04/19 17:52:57 INFO BaseDistributedDataScan: Planning file tasks locally for table nessie.silver.taxi_zone_lookup
26/04/19 17:52:57 INFO SnapshotScan: Sc

PULocationID not found in zone lookup: 0


26/04/19 17:53:00 INFO SparkPartitioningAwareScan: Reporting UnknownPartitioning with 1 partition(s) for table nessie.silver.taxi_zone_lookup
26/04/19 17:53:03 INFO SnapshotScan: Scanning table nessie.silver.taxi_zone_lookup snapshot 2672599193303667998 created at 2026-04-19T17:25:47.429+00:00 with filter true
26/04/19 17:53:03 INFO BaseDistributedDataScan: Planning file tasks locally for table nessie.silver.taxi_zone_lookup


+-----------+
|location_id|
+-----------+
+-----------+



26/04/19 17:53:03 INFO SnapshotScan: Scanning table nessie.silver.yellow_taxi snapshot 5830921864270721079 created at 2026-04-19T17:19:17.176+00:00 with filter true
26/04/19 17:53:03 INFO BaseDistributedDataScan: Planning file tasks locally for table nessie.silver.yellow_taxi
26/04/19 17:53:03 INFO SparkPartitioningAwareScan: Reporting UnknownPartitioning with 8 partition(s) for table nessie.silver.yellow_taxi
26/04/19 17:53:03 INFO SnapshotScan: Scanning table nessie.silver.taxi_zone_lookup snapshot 2672599193303667998 created at 2026-04-19T17:25:47.429+00:00 with filter location_id IS NOT NULL
26/04/19 17:53:03 INFO BaseDistributedDataScan: Planning file tasks locally for table nessie.silver.taxi_zone_lookup
26/04/19 17:53:03 INFO SparkPartitioningAwareScan: Reporting UnknownPartitioning with 1 partition(s) for table nessie.silver.taxi_zone_lookup
26/04/19 17:53:04 INFO SnapshotScan: Scanning table nessie.silver.taxi_zone_lookup snapshot 2672599193303667998 created at 2026-04-19T17:2

DOLocationID not found in zone lookup: 0


26/04/19 17:53:05 INFO SparkPartitioningAwareScan: Reporting UnknownPartitioning with 8 partition(s) for table nessie.silver.yellow_taxi
26/04/19 17:53:05 INFO SnapshotScan: Scanning table nessie.silver.taxi_zone_lookup snapshot 2672599193303667998 created at 2026-04-19T17:25:47.429+00:00 with filter location_id IS NOT NULL
26/04/19 17:53:05 INFO BaseDistributedDataScan: Planning file tasks locally for table nessie.silver.taxi_zone_lookup
26/04/19 17:53:05 INFO SparkPartitioningAwareScan: Reporting UnknownPartitioning with 1 partition(s) for table nessie.silver.taxi_zone_lookup


+-----------+
|location_id|
+-----------+
+-----------+



In [29]:
# Muestra
zones.orderBy("location_id").show(10, truncate=False)


26/04/19 17:53:06 INFO SnapshotScan: Scanning table nessie.silver.taxi_zone_lookup snapshot 2672599193303667998 created at 2026-04-19T17:25:47.429+00:00 with filter true
26/04/19 17:53:06 INFO BaseDistributedDataScan: Planning file tasks locally for table nessie.silver.taxi_zone_lookup
26/04/19 17:53:06 INFO SparkPartitioningAwareScan: Reporting UnknownPartitioning with 1 partition(s) for table nessie.silver.taxi_zone_lookup


+-----------+-------------+-----------------------+------------+
|location_id|borough      |zone                   |service_zone|
+-----------+-------------+-----------------------+------------+
|1          |EWR          |Newark Airport         |EWR         |
|2          |Queens       |Jamaica Bay            |Boro Zone   |
|3          |Bronx        |Allerton/Pelham Gardens|Boro Zone   |
|4          |Manhattan    |Alphabet City          |Yellow Zone |
|5          |Staten Island|Arden Heights          |Boro Zone   |
|6          |Staten Island|Arrochar/Fort Wadsworth|Boro Zone   |
|7          |Queens       |Astoria                |Boro Zone   |
|8          |Queens       |Astoria Park           |Boro Zone   |
|9          |Queens       |Auburndale             |Boro Zone   |
|10         |Queens       |Baisley Park           |Boro Zone   |
+-----------+-------------+-----------------------+------------+
only showing top 10 rows

